In [2]:
import os
import sys

# Move up one level to the project root directory and add it to the Python path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import pandas as pd
import numpy as np
from src.data_loader import load_insurance_data
from src.hypothesis_tests import test_claim_frequency, test_numerical_kpi

In [4]:
df = load_insurance_data("../data/insurance_data.csv")

2026-06-17 12:22:37,296 - INFO - Initiating structural data ingestion from: ../data/insurance_data.csv
2026-06-17 12:22:37,357 - INFO - Calculating derived insurance portfolio risk metrics...
2026-06-17 12:22:37,536 - INFO - Ingestion successful. Row count: 10000, Feature columns: 23


In [5]:
if 'Claimed' not in df.columns or df['Claimed'].dtype == object:
    df['Claimed'] = (df['ClaimAmount'] > 0).astype(int)

results_log = []

# --- Hypothesis 1: Provinces vs Risk (Claim Frequency) ---
# Group A: Western Cape, Group B: Gauteng
h1_res = test_claim_frequency(df, 'Province', 'Western Cape', 'Gauteng')
results_log.append({
    "Hypothesis": "H0: No risk (frequency) differences across provinces",
    "KPI Metric": "Claim Frequency",
    "Test Applied": h1_res["Test Type"],
    "p-value": h1_res["p-value"],
    "Result Decision": h1_res["Decision"]
})

# --- Hypothesis 2: ZipCode vs Risk (Claim Severity) ---
# Pick the top 2 highest volume ZipCodes in your output matrix
top_zips = df['ZipCode'].value_counts().index[:2]
h2_res = test_numerical_kpi(df, 'ZipCode', top_zips[0], top_zips[1], 'ClaimAmount', subset_claims_only=True)
results_log.append({
    "Hypothesis": "H0: No claim severity risk differences between zip codes",
    "KPI Metric": "Claim Severity",
    "Test Applied": h2_res["Test Type"],
    "p-value": h2_res["p-value"],
    "Result Decision": h2_res["Decision"]
})

# --- Hypothesis 3: ZipCode vs Profit Margin ---
h3_res = test_numerical_kpi(df, 'ZipCode', top_zips[0], top_zips[1], 'Margin', subset_claims_only=False)
results_log.append({
    "Hypothesis": "H0: No profit margin difference between zip codes",
    "KPI Metric": "Net Margin Contribution",
    "Test Applied": h3_res["Test Type"],
    "p-value": h3_res["p-value"],
    "Result Decision": h3_res["Decision"]
})

# --- Hypothesis 4: Gender vs Risk (Claim Frequency) ---
h4_res = test_claim_frequency(df, 'Gender', 'Male', 'Female')
results_log.append({
    "Hypothesis": "H0: No significant risk difference between Women and Men",
    "KPI Metric": "Claim Frequency",
    "Test Applied": h4_res["Test Type"],
    "p-value": h4_res["p-value"],
    "Result Decision": h4_res["Decision"]
})

ValueError: No data; `observed` has size 0.